In [10]:
import cv2
import mediapipe as mp
import math
import csv
import os
import json
from datetime import datetime

In [2]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

In [3]:

def calculate_angle(a, b, c):
    angle = math.degrees(
        math.atan2(c[1] - b[1], c[0] - b[0])
        - math.atan2(a[1] - b[1], a[0] - b[0])
    )

    angle = abs(angle)

    if angle > 180:
        angle = 360 - angle

    return angle

In [4]:
# Convert landmark to pixel coordinates
def get_point(landmarks, landmark, width, height):
    point = landmarks[landmark]
    return (
        int(point.x * width),
        int(point.y * height)
    )


In [5]:
def config_to_bool(value):
    #Convert JSON values into True or False

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return value != 0

    if isinstance(value, str):
        return value.strip().lower() in (
            "true", "yes", "y", "1"
        )

    return False

In [19]:
config_file = "safelift_config.json"

DEFAULT_CONFIG = {
    "task_name": "Repetitive ground-level lifting",
    "load_weight_kg": 15.0,
    "shock_or_rapid_force": False,
    "coupling_quality": "good",

    "trunk_twist": False,
    "trunk_side_bending": False,
    "neck_twist": False,
    "neck_side_bending": False,

    "wrist_angle_deg": 0,
    "wrist_twist": False,
    "arm_abducted": False,
    "shoulder_raised": False,
    "arm_supported": False,
    "one_leg_weight_bearing": False,

    "static_posture_over_1_minute": False,
    "repetitive_small_range_actions": False,
    "rapid_posture_changes_or_unstable_base": False
}

if not os.path.exists(config_file):

    with open(config_file, "w", encoding="utf-8") as file:
        json.dump(DEFAULT_CONFIG, file, indent=4)

    print("Default configuration created.")

with open(config_file, "r", encoding="utf-8") as file:
    config = json.load(file)

# Add missing settings without overwriting existing values.
for key, value in DEFAULT_CONFIG.items():
    config.setdefault(key, value)

task_name = config["task_name"]
load_kg = float(config["load_weight_kg"])
shock_force = config_to_bool(
    config["shock_or_rapid_force"]
)
coupling_quality = str(
    config["coupling_quality"]
).strip().lower()

trunk_twist = config_to_bool(config["trunk_twist"])
trunk_side_bending = config_to_bool(
    config["trunk_side_bending"]
)
neck_twist = config_to_bool(config["neck_twist"])
neck_side_bending = config_to_bool(
    config["neck_side_bending"]
)

wrist_angle = float(config["wrist_angle_deg"])
wrist_twist = config_to_bool(config["wrist_twist"])
arm_abducted = config_to_bool(config["arm_abducted"])
shoulder_raised = config_to_bool(config["shoulder_raised"])
arm_supported = config_to_bool(config["arm_supported"])

one_leg_weight_bearing = config_to_bool(
    config["one_leg_weight_bearing"]
)

static_posture = config_to_bool(
    config["static_posture_over_1_minute"]
)

repetitive_action = config_to_bool(
    config["repetitive_small_range_actions"]
)

rapid_or_unstable = config_to_bool(
    config["rapid_posture_changes_or_unstable_base"]
)

if load_kg < 0:
    raise ValueError("Load weight cannot be negative.")

if coupling_quality not in (
    "good", "fair", "poor", "unacceptable"
):
    raise ValueError(
        "Coupling must be good, fair, poor, or unacceptable."
    )

print("Task:", task_name)
print("Load:", load_kg, "kg")
print("Coupling:", coupling_quality)
print("Configuration loaded successfully.")

Task: Repetitive ground-level lifting
Load: 15.0 kg
Coupling: good
Configuration loaded successfully.


In [20]:
csv_file = "safelift_reba_records.csv"
screenshot_folder = "safelift_screenshots"

os.makedirs(screenshot_folder, exist_ok=True)

stage_names = [
    "01_upright",
    "02_start_bend",
    "03_middle_bend",
    "04_lowest",
    "05_return"
]

headers = [
    "Time",
    "Cycle_Number",
    "Max_Torso_Angle_Deg",
    "Min_Knee_Angle_Deg",
    "Neck_Angle_Deg",
    "Upper_Arm_Angle_Deg",
    "Elbow_Angle_Deg",
    "Cycle_Duration_Seconds",
    "REBA_Score",
    "REBA_Action_Level"
]

if not os.path.exists(csv_file):

    with open(
        csv_file,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.writer(file)
        writer.writerow(headers)

    print("New CSV created.")

else:

    with open(
        csv_file,
        "r",
        newline="",
        encoding="utf-8"
    ) as file:

        reader = csv.reader(file)
        existing_headers = next(reader, [])

    if existing_headers != headers:
        raise ValueError(
            "The existing CSV headers do not match. "
            "Rename the old CSV before running this version."
        )

    print("CSV headers are correct.")

print("CSV file:", os.path.abspath(csv_file))
print("Screenshot folder:", os.path.abspath(screenshot_folder))

CSV headers are correct.
CSV file: c:\Users\akhil\Downloads\ML\DL\Media_pipe\project_safelift\safelift_reba_records.csv
Screenshot folder: c:\Users\akhil\Downloads\ML\DL\Media_pipe\project_safelift\safelift_screenshots


In [21]:
BEND_ANGLE = 45
RETURN_ANGLE = 20

KNEE_BEND_ANGLE = 145
UPRIGHT_KNEE_ANGLE = 170

UPRIGHT_CONFIRM_FRAMES = 5
MOVEMENT_CONFIRM_FRAMES = 5

MIN_CYCLE_SECONDS = 0.8

lift_count = 0

cycle_active = False
cycle_start_time = None
ready_for_lift = False

movement_frames = 0
upright_return_frames = 0

upright_frame = None
early_bend_frame = None
middle_bend_frame = None
lowest_frame = None
return_frame = None

# Smooth the measured torso and knee angles.
torso_history = []
knee_history = []

# Measurements for the current lift.
max_torso_angle = 0.0
min_knee_angle = 180.0

max_neck_angle = 0.0
max_upper_arm_angle = 0.0
max_elbow_angle = 0.0

lowest_angle_seen = 0.0

In [22]:

# REBA Table A: Trunk, Neck and Legs

TABLE_A = {
    1: {
        1: [1, 2, 3, 4],
        2: [1, 2, 3, 4],
        3: [3, 3, 5, 6]
    },
    2: {
        1: [2, 3, 4, 5],
        2: [3, 4, 5, 6],
        3: [4, 5, 6, 7]
    },
    3: {
        1: [2, 4, 5, 6],
        2: [4, 5, 6, 7],
        3: [5, 6, 7, 8]
    },
    4: {
        1: [3, 5, 6, 7],
        2: [5, 6, 7, 8],
        3: [6, 7, 8, 9]
    },
    5: {
        1: [4, 6, 7, 8],
        2: [6, 7, 8, 9],
        3: [7, 8, 9, 9]
    }
}


# REBA Table B: Arms and Wrist

TABLE_B = {
    1: {
        1: [1, 2, 2],
        2: [1, 2, 3]
    },
    2: {
        1: [1, 2, 3],
        2: [2, 3, 4]
    },
    3: {
        1: [3, 4, 5],
        2: [4, 5, 5]
    },
    4: {
        1: [4, 5, 5],
        2: [5, 6, 7]
    },
    5: {
        1: [6, 7, 8],
        2: [7, 8, 8]
    },
    6: {
        1: [7, 8, 8],
        2: [8, 9, 9]
    }
}


# REBA Table C: Combines Score A and Score B

TABLE_C = [
    [1, 1, 1, 2, 3, 3, 4, 5, 6, 7, 7, 7],
    [1, 2, 2, 3, 4, 4, 5, 6, 6, 7, 7, 8],
    [2, 3, 3, 3, 4, 5, 6, 7, 7, 8, 8, 8],
    [3, 4, 4, 4, 5, 6, 7, 8, 8, 9, 9, 9],
    [4, 4, 4, 5, 6, 7, 8, 8, 9, 9, 9, 9],
    [6, 6, 6, 7, 8, 8, 9, 9, 10, 10, 10, 10],
    [7, 7, 7, 8, 9, 9, 9, 10, 10, 11, 11, 11],
    [8, 8, 8, 9, 10, 10, 10, 10, 10, 11, 11, 11],
    [9, 9, 9, 10, 10, 10, 11, 11, 11, 12, 12, 12],
    [10, 10, 10, 11, 11, 11, 11, 12, 12, 12, 12, 12],
    [11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12],
    [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
]


def calculate_reba(
    torso_angle,
    knee_angle,
    neck_angle,
    upper_arm_angle,
    elbow_angle,
    load_kg,
    shock_force,
    coupling_quality,
    trunk_twist=False,
    trunk_side_bending=False,
    neck_twist=False,
    neck_side_bending=False,
    wrist_angle=0,
    wrist_twist=False,
    arm_abducted=False,
    shoulder_raised=False,
    arm_supported=False,
    one_leg_weight_bearing=False,
    static_posture=False,
    repetitive_action=False,
    rapid_or_unstable=False
):
    # Convert the input values into numbers
    torso_angle = abs(float(torso_angle))
    knee_angle = float(knee_angle)
    neck_angle = abs(float(neck_angle))
    upper_arm_angle = abs(float(upper_arm_angle))
    elbow_angle = abs(float(elbow_angle))
    wrist_angle = abs(float(wrist_angle))
    load_kg = float(load_kg)

    # Calculate the trunk score
    if torso_angle <= 5:
        trunk_score = 1
    elif torso_angle <= 20:
        trunk_score = 2
    elif torso_angle <= 60:
        trunk_score = 3
    else:
        trunk_score = 4

    if config_to_bool(trunk_twist):
        trunk_score += 1

    if config_to_bool(trunk_side_bending):
        trunk_score += 1

    trunk_score = min(trunk_score, 5)

    # Calculate the neck score
    if neck_angle <= 20:
        neck_score = 1
    else:
        neck_score = 2

    if config_to_bool(neck_twist):
        neck_score += 1

    if config_to_bool(neck_side_bending):
        neck_score += 1

    neck_score = min(neck_score, 3)

    # Calculate the leg score
    knee_flexion = max(0, 180 - knee_angle)
    leg_score = 1

    if knee_flexion > 60:
        leg_score += 2
    elif knee_flexion >= 30:
        leg_score += 1

    if config_to_bool(one_leg_weight_bearing):
        leg_score += 1

    leg_score = min(leg_score, 4)

    # Find the posture score from Table A
    posture_score_a = TABLE_A[trunk_score][neck_score][leg_score - 1]

    # Add the load adjustment
    if load_kg < 5:
        load_score = 0
    elif load_kg <= 10:
        load_score = 1
    else:
        load_score = 2

    if config_to_bool(shock_force):
        load_score += 1

    score_a = min(posture_score_a + load_score, 12)

    # Calculate the upper-arm score
    if upper_arm_angle <= 20:
        upper_arm_score = 1
    elif upper_arm_angle <= 45:
        upper_arm_score = 2
    elif upper_arm_angle <= 90:
        upper_arm_score = 3
    else:
        upper_arm_score = 4

    if config_to_bool(arm_abducted):
        upper_arm_score += 1

    if config_to_bool(shoulder_raised):
        upper_arm_score += 1

    if config_to_bool(arm_supported):
        upper_arm_score -= 1

    upper_arm_score = max(1, min(upper_arm_score, 6))

    # Calculate the lower-arm score
    if 60 <= elbow_angle <= 100:
        lower_arm_score = 1
    else:
        lower_arm_score = 2

    # Calculate the wrist score
    wrist_score = 1 if wrist_angle <= 15 else 2

    if config_to_bool(wrist_twist):
        wrist_score += 1

    wrist_score = min(wrist_score, 3)

    # Find the posture score from Table B
    posture_score_b = TABLE_B[upper_arm_score][lower_arm_score][wrist_score - 1]

    # Add the coupling adjustment
    coupling_scores = {
        "good": 0,
        "fair": 1,
        "poor": 2,
        "unacceptable": 3
    }

    coupling_key = str(coupling_quality).strip().lower()

    if coupling_key not in coupling_scores:
        raise ValueError(
            "Coupling must be good, fair, poor, or unacceptable."
        )

    score_b = min(
        posture_score_b + coupling_scores[coupling_key],
        12
    )

    # Combine scores using Table C
    score_c = TABLE_C[score_a - 1][score_b - 1]

    # Calculate the activity score
    activity_score = (
        int(config_to_bool(static_posture))
        + int(config_to_bool(repetitive_action))
        + int(config_to_bool(rapid_or_unstable))
    )

    final_score = min(score_c + activity_score, 15)

    # Assign the action level
    if final_score == 1:
        action_level = "Negligible"
    elif final_score <= 3:
        action_level = "Low"
    elif final_score <= 7:
        action_level = "Medium"
    elif final_score <= 10:
        action_level = "High"
    else:
        action_level = "Very High"

    return final_score, action_level


# Check that Table C contains 12 rows and 12 columns
assert len(TABLE_C) == 12
assert all(len(row) == 12 for row in TABLE_C)

print("REBA tables loaded.")
print("Scoring function is ready.")


REBA tables loaded.
Scoring function is ready.


In [23]:
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError("Could not open webcam.")

print("SafeLift AI started.")
print("Stand sideways with your full body visible.")
print("Bend down and return upright to complete a lift.")
print("Press Q to quit.")

SafeLift AI started.
Stand sideways with your full body visible.
Bend down and return upright to complete a lift.
Press Q to quit.


In [24]:
try:

    with mp_pose.Pose(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:

        while True:

            success, frame = camera.read()

            if not success:
                print("Could not read webcam.")
                break

            frame = cv2.flip(frame, 1)

            height, width, _ = frame.shape

            rgb_frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            result = pose.process(rgb_frame)

            status = "No person detected"

            if result.pose_landmarks:

                landmarks = result.pose_landmarks.landmark

                mp_draw.draw_landmarks(
                    frame,
                    result.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS
                )

                required_points = [
                    mp_pose.PoseLandmark.LEFT_SHOULDER,
                    mp_pose.PoseLandmark.LEFT_HIP,
                    mp_pose.PoseLandmark.LEFT_KNEE,
                    mp_pose.PoseLandmark.LEFT_ANKLE,
                    mp_pose.PoseLandmark.LEFT_EAR,
                    mp_pose.PoseLandmark.LEFT_ELBOW,
                    mp_pose.PoseLandmark.LEFT_WRIST
                ]

                valid_pose = all(
                    landmarks[p].visibility > 0.5
                    for p in required_points
                )

                if valid_pose:

                    # Get body coordinates.

                    shoulder = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_SHOULDER,
                        width,
                        height
                    )

                    hip = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_HIP,
                        width,
                        height
                    )

                    knee = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_KNEE,
                        width,
                        height
                    )

                    ankle = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_ANKLE,
                        width,
                        height
                    )

                    ear = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_EAR,
                        width,
                        height
                    )

                    elbow = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_ELBOW,
                        width,
                        height
                    )

                    wrist = get_point(
                        landmarks,
                        mp_pose.PoseLandmark.LEFT_WRIST,
                        width,
                        height
                    )

                    # Calculate knee angle.

                    raw_knee_angle = calculate_angle(
                        hip,
                        knee,
                        ankle
                    )

                    # Calculate torso inclination from vertical.

                    dx = shoulder[0] - hip[0]
                    dy = hip[1] - shoulder[1]

                    raw_torso_angle = math.degrees(
                        math.atan2(
                            abs(dx),
                            max(abs(dy), 1)
                        )
                    )

                    # Calculate neck angle.

                    raw_neck_angle = calculate_angle(
                        ear,
                        shoulder,
                        hip
                    )

                    raw_neck_flexion = abs(
                        180.0 - raw_neck_angle
                    )

                    # Calculate upper-arm angle relative to trunk.

                    raw_upper_arm_angle = calculate_angle(
                        hip,
                        shoulder,
                        elbow
                    )

                    # Calculate internal elbow angle.

                    raw_elbow_angle = calculate_angle(
                        shoulder,
                        elbow,
                        wrist
                    )

                    # Smooth torso and knee angles.

                    torso_history.append(raw_torso_angle)
                    knee_history.append(raw_knee_angle)

                    if len(torso_history) > 5:
                        torso_history.pop(0)

                    if len(knee_history) > 5:
                        knee_history.pop(0)

                    torso_angle = sorted(torso_history)[
                        len(torso_history) // 2
                    ]

                    knee_angle = sorted(knee_history)[
                        len(knee_history) // 2
                    ]

                    neck_angle = raw_neck_flexion
                    upper_arm_angle = raw_upper_arm_angle
                    elbow_angle = raw_elbow_angle

                    # MOVEMENT DETECTION

                    movement_started = (
                        torso_angle >= BEND_ANGLE
                        or knee_angle <= KNEE_BEND_ANGLE
                    )

                    upright_now = (
                        torso_angle < RETURN_ANGLE
                        and knee_angle >= UPRIGHT_KNEE_ANGLE
                    )

                    if upright_now:
                        status = "UPRIGHT"

                    elif movement_started:
                        status = "BENDING / SQUATTING"

                    elif torso_angle < BEND_ANGLE:
                        status = "STARTING MOVEMENT"

                    else:
                        status = "MOVING"

                    # Prepare to detect a new lift.

                    if not cycle_active and upright_now:

                        ready_for_lift = True

                        if upright_frame is None:
                            upright_frame = frame.copy()

                    # Confirm movement over several frames.

                    if (
                        not cycle_active
                        and ready_for_lift
                        and movement_started
                    ):

                        movement_frames += 1

                    elif not cycle_active:

                        movement_frames = 0

                    # Start a new lift cycle.

                    if (
                        ready_for_lift
                        and not cycle_active
                        and movement_frames >= MOVEMENT_CONFIRM_FRAMES
                    ):

                        cycle_active = True
                        cycle_start_time = datetime.now()

                        max_torso_angle = torso_angle
                        min_knee_angle = knee_angle

                        max_neck_angle = neck_angle
                        max_upper_arm_angle = upper_arm_angle
                        max_elbow_angle = elbow_angle

                        lowest_angle_seen = torso_angle

                        early_bend_frame = frame.copy()
                        middle_bend_frame = None
                        lowest_frame = frame.copy()
                        return_frame = None

                        upright_return_frames = 0
                        movement_frames = 0
                        ready_for_lift = False

                        print("Lift started.")

                    # Track measurements during the active lift.

                    if cycle_active:

                        max_torso_angle = max(
                            max_torso_angle,
                            torso_angle
                        )

                        min_knee_angle = min(
                            min_knee_angle,
                            knee_angle
                        )

                        max_neck_angle = max(
                            max_neck_angle,
                            neck_angle
                        )

                        max_upper_arm_angle = max(
                            max_upper_arm_angle,
                            upper_arm_angle
                        )

                        max_elbow_angle = max(
                            max_elbow_angle,
                            elbow_angle
                        )

                        # Capture the middle bending stage.

                        if (
                            55 <= torso_angle < 70
                            and middle_bend_frame is None
                        ):

                            middle_bend_frame = frame.copy()

                        # Capture the deepest torso bend.

                        if torso_angle > lowest_angle_seen:

                            lowest_angle_seen = torso_angle
                            lowest_frame = frame.copy()

                        # Confirm return to upright.

                        if upright_now:
                            upright_return_frames += 1

                        else:
                            upright_return_frames = 0

                        # Complete the lift after upright confirmation.

                        if (
                            upright_return_frames
                            >= UPRIGHT_CONFIRM_FRAMES
                        ):

                            cycle_end_time = datetime.now()

                            cycle_duration = (
                                cycle_end_time - cycle_start_time
                            ).total_seconds()

                            return_frame = frame.copy()

                            if cycle_duration >= MIN_CYCLE_SECONDS:

                                # Prepare five screenshots.

                                frames_to_save = [
                                    upright_frame,
                                    early_bend_frame,
                                    middle_bend_frame,
                                    lowest_frame,
                                    return_frame
                                ]

                                frames_to_save = [
                                    image if image is not None
                                    else frame.copy()
                                    for image in frames_to_save
                                ]

                                next_cycle_number = lift_count + 1

                                cycle_dir = os.path.join(
                                    screenshot_folder,
                                    f"cycle_{next_cycle_number:03d}"
                                )

                                os.makedirs(
                                    cycle_dir,
                                    exist_ok=True
                                )

                                # Save screenshots.

                                for i, image in enumerate(
                                    frames_to_save
                                ):

                                    image_path = os.path.join(
                                        cycle_dir,
                                        stage_names[i] + ".jpg"
                                    )

                                    if not cv2.imwrite(
                                        image_path,
                                        image
                                    ):

                                        print(
                                            "Could not save screenshot:",
                                            image_path
                                        )

                                # Calculate provisional REBA score.

                                reba_score, reba_action_level = (
                                    calculate_reba(
                                        torso_angle=max_torso_angle,
                                        knee_angle=min_knee_angle,
                                        neck_angle=max_neck_angle,
                                        upper_arm_angle=max_upper_arm_angle,
                                        elbow_angle=max_elbow_angle,
                                        load_kg=load_kg,
                                        shock_force=shock_force,
                                        coupling_quality=coupling_quality,
                                        trunk_twist=trunk_twist,
                                        trunk_side_bending=trunk_side_bending,
                                        neck_twist=neck_twist,
                                        neck_side_bending=neck_side_bending,
                                        wrist_angle=wrist_angle,
                                        wrist_twist=wrist_twist,
                                        arm_abducted=arm_abducted,
                                        shoulder_raised=shoulder_raised,
                                        arm_supported=arm_supported,
                                        one_leg_weight_bearing=one_leg_weight_bearing,
                                        static_posture=static_posture,
                                        repetitive_action=repetitive_action,
                                        rapid_or_unstable=rapid_or_unstable
                                    )
                                )

                                lift_count += 1

                                # Write the completed lift to CSV.

                                with open(
                                    csv_file,
                                    "a",
                                    newline="",
                                    encoding="utf-8"
                                ) as file:

                                    writer = csv.writer(file)

                                    writer.writerow([
                                        cycle_end_time.strftime(
                                            "%Y-%m-%d %H:%M:%S"
                                        ),
                                        lift_count,
                                        round(max_torso_angle, 1),
                                        round(min_knee_angle, 1),
                                        round(max_neck_angle, 1),
                                        round(max_upper_arm_angle, 1),
                                        round(max_elbow_angle, 1),
                                        round(cycle_duration, 2),
                                        reba_score,
                                        reba_action_level
                                    ])

                                # Display results.

                                print("\nLift completed:", lift_count)

                                print(
                                    "Maximum torso angle:",
                                    round(max_torso_angle, 1)
                                )

                                print(
                                    "Minimum knee angle:",
                                    round(min_knee_angle, 1)
                                )

                                print(
                                    "Neck angle estimate:",
                                    round(max_neck_angle, 1)
                                )

                                print(
                                    "Upper-arm angle:",
                                    round(max_upper_arm_angle, 1)
                                )

                                print(
                                    "Maximum elbow angle:",
                                    round(max_elbow_angle, 1)
                                )

                                print(
                                    "Cycle duration:",
                                    round(cycle_duration, 2),
                                    "seconds"
                                )

                                print(
                                    "Provisional REBA score:",
                                    reba_score
                                )

                                print(
                                    "Provisional action level:",
                                    reba_action_level
                                )

                                print("Screenshots:", cycle_dir)
                                print("CSV:", os.path.abspath(csv_file))

                            else:

                                print(
                                    "Ignored short movement:",
                                    round(cycle_duration, 2),
                                    "seconds"
                                )

                            # Reset cycle state.

                            cycle_active = False
                            cycle_start_time = None

                            ready_for_lift = True
                            movement_frames = 0
                            upright_return_frames = 0

                            upright_frame = frame.copy()

                            early_bend_frame = None
                            middle_bend_frame = None
                            lowest_frame = None
                            return_frame = None

                            max_torso_angle = 0.0
                            min_knee_angle = 180.0

                            max_neck_angle = 0.0
                            max_upper_arm_angle = 0.0
                            max_elbow_angle = 0.0

                            lowest_angle_seen = 0.0

                    # Display live measurements.

                    display_lines = [
                        f"Torso: {torso_angle:.1f} deg",
                        f"Knee: {knee_angle:.1f} deg",
                        f"Neck: {neck_angle:.1f} deg",
                        f"Upper arm: {upper_arm_angle:.1f} deg",
                        f"Elbow: {elbow_angle:.1f} deg",
                        f"Stage: {status}",
                        f"Lifts: {lift_count}"
                    ]

                    for i, line in enumerate(display_lines):

                        cv2.putText(
                            frame,
                            line,
                            (20, 30 + i * 30),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,
                            (0, 255, 0) if i == 0
                            else (255, 255, 0),
                            2
                        )

                else:

                    cv2.putText(
                        frame,
                        "Move into full camera view",
                        (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 0, 255),
                        2
                    )

            # Display the webcam frame.

            cv2.imshow("SafeLift AI", frame)

            # Press Q to stop.

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

finally:

    camera.release()
    cv2.destroyAllWindows()

c:\Users\akhil\Downloads\ML\DL\Media_pipe\safelift_env\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Lift started.

Lift completed: 1
Maximum torso angle: 86.5
Minimum knee angle: 145.4
Neck angle estimate: 23.1
Upper-arm angle: 103.9
Maximum elbow angle: 168.5
Cycle duration: 2.62 seconds
Provisional REBA score: 10
Provisional action level: High
Screenshots: safelift_screenshots\cycle_001
CSV: c:\Users\akhil\Downloads\ML\DL\Media_pipe\project_safelift\safelift_reba_records.csv
Lift started.

Lift completed: 2
Maximum torso angle: 85.9
Minimum knee angle: 139.0
Neck angle estimate: 28.6
Upper-arm angle: 107.3
Maximum elbow angle: 160.7
Cycle duration: 2.18 seconds
Provisional REBA score: 10
Provisional action level: High
Screenshots: safelift_screenshots\cycle_002
CSV: c:\Users\akhil\Downloads\ML\DL\Media_pipe\project_safelift\safelift_reba_records.csv
Lift started.

Lift completed: 3
Maximum torso angle: 85.8
Minimum knee angle: 133.3
Neck angle estimate: 23.5
Upper-arm angle: 102.1
Maximum elbow angle: 162.7
Cycle duration: 1.82 seconds
Provisional REBA score: 10
Provisional action 

In [29]:
print("\nSafeLift AI stopped.")
print("Completed lifts in this run:", lift_count)
print("CSV file:", os.path.abspath(csv_file))

import pandas as pd

df = pd.read_csv(csv_file)

print("Number of columns:", len(df.columns))
print("Column names:", df.columns.tolist())

display(df.tail(6))


SafeLift AI stopped.
Completed lifts in this run: 6
CSV file: c:\Users\akhil\Downloads\ML\DL\Media_pipe\project_safelift\safelift_reba_records.csv
Number of columns: 10
Column names: ['Time', 'Cycle_Number', 'Max_Torso_Angle_Deg', 'Min_Knee_Angle_Deg', 'Neck_Angle_Deg', 'Upper_Arm_Angle_Deg', 'Elbow_Angle_Deg', 'Cycle_Duration_Seconds', 'REBA_Score', 'REBA_Action_Level']


,Time,Cycle_Number,Max_Torso_Angle_Deg,Min_Knee_Angle_Deg,Neck_Angle_Deg,Upper_Arm_Angle_Deg,Elbow_Angle_Deg,Cycle_Duration_Seconds,REBA_Score,REBA_Action_Level
17,2026-10-18 21:21:04,1,86.5,145.4,23.1,103.9,168.5,2.62,10,High
18,2026-10-18 21:21:10,2,85.9,139.0,28.6,107.3,160.7,2.18,10,High
19,2026-10-18 21:21:15,3,85.8,133.3,23.5,102.1,162.7,1.82,10,High
20,2026-10-18 21:21:19,4,61.9,172.4,22.4,64.4,159.7,1.44,8,High
21,2026-10-18 21:21:24,5,48.2,171.9,22.6,47.8,151.4,1.06,7,Medium
22,2026-10-18 21:21:28,6,55.4,171.8,27.0,46.8,141.1,1.20,7,Medium


In [9]:
import tensorflow as tf
import streamlit as st
import google.protobuf

print("TensorFlow:", tf.__version__)
print("Streamlit:", st.__version__)
print("Protobuf:", google.protobuf.__version__)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)

TensorFlow: 2.15.1
Streamlit: 1.66.1
Protobuf: 7.36.2
OpenCV: 4.10.0
MediaPipe: 0.10.9
